In [0]:
# Cria o schema e volume no catálogo workspace
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.olist")

spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.olist.raw_data
""")

print("Pronto!")

In [0]:
#Cria os vaolumes para cada camada da arquitetura Medallion
volumes = ["bronze", "silver",  "gold", "dead_letter"]

for volume in volumes:
    spark.sql(f"""
        CREATE VOLUME IF NOT EXISTS workspace.olist.{volume}
    """)
print("\nEstrutura completa!")

In [0]:
#Lista os arquivos no volume
import os
for f in dbutils.fs.ls("/Volumes/workspace/olist/raw_data/"):
    tamanho_mb = f.size/ (1024*1024)
    print(f"{f.name} - {tamanho_mb:.2f} MB")
#


In [0]:
from pyspark.sql import functions as F
import datetime


# Lista de arquivos para ingerir
tabelas = [
    ("olist_orders_dataset.csv", "orders"),
    ("olist_order_items_dataset.csv", "orders_items"),
    ("olist_customers_dataset.csv", "customers"),
    ("olist_products_dataset.csv", "products")
]

for arquivo, nome in tabelas:
    #Lê o CSV
    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
            .option("mergeSchema", "true") \
            .load(f"/Volumes/workspace/olist/raw_data/{arquivo}", format="csv")

    # Adiciona colunas de auditoria
    df_bronze = df \
        .withColumn("_ingest_timestamp", F.now()) \
        .withColumn("_source_filename",  F.col("_metadata.file_path")) \
        .withColumn("_processing_date",  F.lit(str(datetime.date.today())))

    # Grava como Delta na camada Bronze
    df_bronze.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"/Volumes/workspace/olist/bronze/{nome}")

    print(f"{nome}: {df_bronze.count()} registros gravados")

print("\nSprint 1 - Camada Bronze criada com sucesso!")

#Teste git